# Notebook 04 — Evaluation
Computes all metrics and generates cross-layer comparison plots.

Metrics:
- Reconstruction MSE and L0 sparsity (from training logs)
- Dead feature rate
- CLIP interpretability score (mean cosine similarity of top label)
- Cross-layer label distribution comparison

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

ckpt4   = torch.load('../checkpoints/sae_layer4.pt',  weights_only=False)
ckpt8   = torch.load('../checkpoints/sae_layer8.pt',  weights_only=False)
ckpt12  = torch.load('../checkpoints/sae_layer12.pt', weights_only=False)
labels4  = torch.load('../checkpoints/labels_layer4.pt',  weights_only=False)
labels8  = torch.load('../checkpoints/labels_layer8.pt',  weights_only=False)
labels12 = torch.load('../checkpoints/labels_layer12.pt', weights_only=False)
raw4   = torch.load('../checkpoints/raw_labels_layer4.pt',  weights_only=False)
raw8   = torch.load('../checkpoints/raw_labels_layer8.pt',  weights_only=False)
raw12  = torch.load('../checkpoints/raw_labels_layer12.pt', weights_only=False)
log4  = ckpt4['log']
log8  = ckpt8['log']
log12 = ckpt12['log']

In [ ]:
def last(log, key): return log[key][-1]

rows = [
    ('Final MSE',         last(log4,'mse'),  last(log8,'mse'),  last(log12,'mse')),
    ('Final L0',          last(log4,'l0'),   last(log8,'l0'),   last(log12,'l0')),
    ('Dead feature %',    last(log4,'dead_pct'), last(log8,'dead_pct'), last(log12,'dead_pct')),
    ('SAE mean CLIP cos', np.mean(labels4['scores']),     np.mean(labels8['scores']),     np.mean(labels12['scores'])),
    ('SAE mean Spearman', np.mean(labels4['corr_scores']),np.mean(labels8['corr_scores']),np.mean(labels12['corr_scores'])),
    ('Raw neuron Spearman',np.mean(raw4['corr_scores']),  np.mean(raw8['corr_scores']),   np.mean(raw12['corr_scores'])),
]
print(f"{'Metric':<28} {'Layer 4':>10} {'Layer 8':>10} {'Layer 12':>10}")
print('-' * 60)
for name, v4, v8, v12 in rows:
    print(f'{name:<28} {v4:>10.4f} {v8:>10.4f} {v12:>10.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for log, label, color in [(log4, 'Layer 4', 'steelblue'), (log8, 'Layer 8', 'seagreen'), (log12, 'Layer 12', 'crimson')]:
    s = log['step']
    axes[0].plot(s, log['mse'],      label=label, color=color)
    axes[1].plot(s, log['l0'],       label=label, color=color)
    axes[2].plot(s, log['dead_pct'], label=label, color=color)
axes[1].axhline(y=5,  color='gray', linestyle='--', linewidth=1)
axes[1].axhline(y=50, color='gray', linestyle='--', linewidth=1)
axes[2].axhline(y=10, color='gray', linestyle='--', linewidth=1)
for ax, title in zip(axes, ['Reconstruction MSE', 'L0 (avg active features)', 'Dead Feature %']):
    ax.set_title(title); ax.set_xlabel('Step'); ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/training_curves.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lbls, title in [(axes[0], labels4, 'Layer 4'),
                         (axes[1], labels8, 'Layer 8'),
                         (axes[2], labels12, 'Layer 12')]:
    top15 = Counter(lbls['labels']).most_common(15)
    words, counts = zip(*top15)
    ax.barh(words, counts)
    ax.set_title(f'{title} — Top Labels')
    ax.set_xlabel('Feature count')
plt.tight_layout()
plt.savefig('../checkpoints/label_distribution.png', dpi=150)
plt.show()

In [ ]:
# SAE features vs raw ViT neurons: Spearman correlation distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sae_lbls, raw_lbls, title in [
        (axes[0], labels4,  raw4,  'Layer 4'),
        (axes[1], labels8,  raw8,  'Layer 8'),
        (axes[2], labels12, raw12, 'Layer 12')]:
    ax.hist(sae_lbls['corr_scores'], bins=30, alpha=0.6, label='SAE features', color='steelblue')
    ax.hist(raw_lbls['corr_scores'], bins=30, alpha=0.6, label='Raw neurons',  color='tomato')
    ax.set_title(f'{title}  SAE={np.mean(sae_lbls["corr_scores"]):.3f}  '
                 f'Raw={np.mean(raw_lbls["corr_scores"]):.3f}')
    ax.set_xlabel('Spearman correlation')
    ax.legend()
plt.suptitle('CLIP Interpretability: SAE Features vs Raw ViT Neurons', fontsize=12)
plt.tight_layout()
plt.savefig('../checkpoints/sae_vs_raw.png', dpi=150)
plt.show()

In [ ]:
from torchvision import datasets
from torchvision import transforms as T
from PIL import Image
from src.sae import SAE
from src.clip_labeler import crop_patch

IMAGENET_VAL_DIR = r"D:\Master Material\XAI\XAI_project\archive\imagenet-val"
N_EVAL_FEATURES  = 50
N_SAMPLE_TOKENS  = 50000

dataset          = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=None)
selected_indices = torch.load('../data/selected_indices.pt', weights_only=True)
pil_transform    = T.Compose([T.Resize(256), T.CenterCrop(224)])

def get_patch(orig_patch_idx, context=64):
    """Retrieve the 64x64 crop for a patch-only token index."""
    img_pos    = orig_patch_idx // 196
    tok_off    = orig_patch_idx % 196 + 1  # original tok_offset 1-196
    actual_idx = int(selected_indices[img_pos])
    img = Image.open(dataset.imgs[actual_idx][0]).convert('RGB')
    img = pil_transform(img)
    return crop_patch(img, tok_off, context)

print(f'Dataset: {len(dataset.imgs)} images, {len(selected_indices)} selected')

In [ ]:
def run_human_eval(acts_path, ckpt_path, labels_data, layer_name):
    acts = torch.load(acts_path, weights_only=False)
    # Filter CLS tokens to match SAE training space
    patch_mask = torch.arange(acts.shape[0]) % 197 != 0
    acts = acts[patch_mask]  # (N_images * 196, 768)

    ckpt = torch.load(ckpt_path, weights_only=False)
    idx  = torch.randperm(acts.shape[0],
                          generator=torch.Generator().manual_seed(42))[:N_SAMPLE_TOKENS]
    acts_sample = (acts[idx] - ckpt['acts_mean']) / ckpt['acts_rms']

    sae = SAE(d_input=768, d_hidden=3072)
    sae.load_state_dict(ckpt['state_dict'])
    sae.eval()
    with torch.no_grad():
        sae_acts, _ = sae(acts_sample)

    torch.manual_seed(0)
    feat_ids = torch.randperm(len(labels_data['labels']))[:N_EVAL_FEATURES].tolist()

    ratings = {}
    for feat_i in feat_ids:
        vals  = sae_acts[:, feat_i]
        top10 = vals.topk(10).indices.tolist()
        # Map sampled positions → original patch-only indices
        orig_patch_indices = [idx[p].item() for p in top10]

        patches = [get_patch(orig_i) for orig_i in orig_patch_indices
                   if orig_i // 196 < len(selected_indices)]

        fig, axes = plt.subplots(2, 5, figsize=(12, 5))
        label_str = labels_data['labels'][feat_i]
        score_str = f'{labels_data["scores"][feat_i]:.3f}'
        fig.suptitle(
            f"{layer_name} | Feature {feat_i} | CLIP: \"{label_str}\" (score={score_str})",
            fontsize=11)
        for ax, patch in zip(axes.flat, patches):
            ax.imshow(patch); ax.axis('off')
        plt.tight_layout(); plt.show()

        while True:
            r = input(f'  Rate feature {feat_i} (1-5): ').strip()
            if r in {'1','2','3','4','5'}:
                ratings[feat_i] = int(r); break
            print('  Please enter 1-5')
        plt.close()

    return ratings

ratings4  = run_human_eval('../data/layer4_activations.pt',
                           '../checkpoints/sae_layer4.pt',  labels4,  'Layer 4')
ratings8  = run_human_eval('../data/layer8_activations.pt',
                           '../checkpoints/sae_layer8.pt',  labels8,  'Layer 8')
ratings12 = run_human_eval('../data/layer12_activations.pt',
                           '../checkpoints/sae_layer12.pt', labels12, 'Layer 12')

In [ ]:
r4  = list(ratings4.values())
r8  = list(ratings8.values())
r12 = list(ratings12.values())
print('Human eval results:')
print(f'  Layer 4  — mean={np.mean(r4):.2f}  std={np.std(r4):.2f}  n={len(r4)}')
print(f'  Layer 8  — mean={np.mean(r8):.2f}  std={np.std(r8):.2f}  n={len(r8)}')
print(f'  Layer 12 — mean={np.mean(r12):.2f}  std={np.std(r12):.2f}  n={len(r12)}')
torch.save({'ratings4': ratings4, 'ratings8': ratings8, 'ratings12': ratings12},
           '../checkpoints/human_ratings.pt')
print('Ratings saved.')